In [12]:
%pip install pyreadr "numpy==1.26.4" "scikit-learn==1.6.1" "joblib==1.5.3" "pandas==2.2.2"

In [13]:
import pyreadr
import pandas as pd
meli = result = pyreadr.read_r("meli_limpio.Rdata")
df = meli['meli']
df.drop(columns = ["preciom2", "condicion_2", "tipo_inmueble_2", "start_time_new",
                   "bajada_new", "Inmueble.1", "stop_time", "start_time", "bajada",
                   "lat", "long", "direccion", "moneda", "operacion", "categ",
                   "titulo", "id"], inplace=True)

In [14]:
df = df[df["sup_tot"] >= 15]


# Entremnamiento del modelo

In [15]:
from sklearn.model_selection import train_test_split

target = "precio"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
variables_categoricas = ["tipo_inmueble", "condicion", "barrio", "terraza", "patio",
                         "toilette", "aircond", "calefacc", "jardin", "kitchenette",
                         "losa_rad", "parrillero", "piscina", "sala_reuniones", "seguridad",
                         "amoblado", "estado", "orientacion", "garage", "comedor"]

In [17]:
variables_numericas = ["banos", "ap_ppiso","dormitorios", "expensas",
                       "sup_constru","antiguedad", "ambientes", "ascensores",
                       "sup_tot"]

badNumericCols = [
    c for c in variables_numericas
    if not pd.api.types.is_numeric_dtype(X_train[c])
]

colsToCoerce = badNumericCols

for c in colsToCoerce:
    X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce")


In [18]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, variables_numericas),
        ("cat", categorical_transformer, variables_categoricas)
    ]
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge())
])

import numpy as np
from sklearn.compose import TransformedTargetRegressor

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge())
])

modeloConLog = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)


In [19]:
param_grid = [
    {
        "regressor__model": [Ridge()],
        "regressor__model__alpha": [0.5, 1.0, 5.0]
    },

    {
        "regressor__model": [RandomForestRegressor(random_state=42, n_jobs=-1)],
        "regressor__model__n_estimators": [100, 200],
        "regressor__model__max_depth": [15, 20, 25],
        "regressor__model__min_samples_leaf": [5, 10],
        "regressor__model__min_samples_split": [10, 20],
        "regressor__model__max_features": ["sqrt"],
        "regressor__model__max_samples": [0.8]
    },

      {
        "regressor__model": [HistGradientBoostingRegressor(
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            scoring="neg_mean_absolute_error"
        )],
        "regressor__model__learning_rate": [0.01, 0.03, 0.05],
        "regressor__model__max_iter": [300, 500],
        "regressor__model__max_depth": [6, 8, None],
        "regressor__model__max_leaf_nodes": [31, 63],
        "regressor__model__min_samples_leaf": [20, 50],
        "regressor__model__l2_regularization": [0.0, 1.0, 5.0],
        "regressor__model__loss": ["absolute_error"]
    }
]

In [20]:
# VERSION MAS SIMPLIFICADA DESPUES DE ITERAR

param_grid = [
    {
        "regressor__model": [Ridge()],
        "regressor__model__alpha": [1.0]
    },
    {
        "regressor__model": [HistGradientBoostingRegressor(
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            scoring="neg_mean_absolute_error"
        )],
        "regressor__model__learning_rate": [0.03],
        "regressor__model__max_iter": [400],
        "regressor__model__max_depth": [6, 8],
        "regressor__model__max_leaf_nodes": [31],
        "regressor__model__min_samples_leaf": [20],
        "regressor__model__l2_regularization": [0.0, 5.0],
        "regressor__model__loss": ["absolute_error"]
    }
]


In [21]:
X_small = X_train.sample(n=200000, random_state=42)
y_small = y_train.loc[X_small.index]

gs = GridSearchCV(
    estimator=modeloConLog,
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2
)

In [22]:
import numpy as np
from sklearn.metrics import mean_absolute_error

median_price = np.median(y_train)

y_pred_median = np.full(shape=len(y_train), fill_value=median_price)

baseline_mae = mean_absolute_error(y_train, y_pred_median)

print("Mediana:", median_price)
print("Baseline MAE (mediana):", baseline_mae)


Mediana: 165000.0
Baseline MAE (mediana): 116765.03316311724


In [23]:
gs.fit(X_small, y_small)

bestPipeline = gs.best_estimator_
bestParams = gs.best_params_
bestScore = gs.best_score_
cv_mae = -bestScore

Fitting 3 folds for each of 5 candidates, totalling 15 fits


/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [24]:
innerPipeline = bestPipeline.regressor

print("MEJOR MODELO SELECCIONADO POR GRIDSEARCH")
print(f"\nMejores parámetros:")
for key, value in bestParams.items():
    print(f"  {key}: {value}")
print(f"\nMejor score (CV MAE): {cv_mae:.2f}")
print(f"Tipo de modelo: {type(innerPipeline.named_steps['model']).__name__}")
print("=" * 60)


MEJOR MODELO SELECCIONADO POR GRIDSEARCH

Mejores parámetros:
  regressor__model: HistGradientBoostingRegressor(early_stopping=True, random_state=42,
                              scoring='neg_mean_absolute_error')
  regressor__model__l2_regularization: 5.0
  regressor__model__learning_rate: 0.03
  regressor__model__loss: absolute_error
  regressor__model__max_depth: 8
  regressor__model__max_iter: 400
  regressor__model__max_leaf_nodes: 31
  regressor__model__min_samples_leaf: 20

Mejor score (CV MAE): 42786.05
Tipo de modelo: HistGradientBoostingRegressor


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import numpy as np

print("ENTRENAMIENTO FINAL Y EVALUACIÓN")

bestPipeline.fit(X_train, y_train)

print("\n1. Validación Cruzada (5-fold) en conjunto de entrenamiento:")
cv_scores_mae = -cross_val_score(
    bestPipeline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
cv_scores_rmse = np.sqrt(-cross_val_score(
    bestPipeline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
))
cv_scores_r2 = cross_val_score(
    bestPipeline,
    X_train,
    y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print(f"   CV MAE: {cv_scores_mae.mean():.2f} (+/- {cv_scores_mae.std() * 2:.2f})")
print(f"   CV RMSE: {cv_scores_rmse.mean():.2f} (+/- {cv_scores_rmse.std() * 2:.2f})")
print(f"   CV R²: {cv_scores_r2.mean():.4f} (+/- {cv_scores_r2.std() * 2:.4f})")

# Evaluar en conjunto de test
y_pred_test = bestPipeline.predict(X_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

print(f"\n3. Métricas en conjunto de TEST:")
print(f"   MAE: {mae_test:.2f}")
print(f"   RMSE: {rmse_test:.2f}")
print(f"   R²: {r2_test:.4f}")


ENTRENAMIENTO FINAL Y EVALUACIÓN


/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)



1. Validación Cruzada (5-fold) en conjunto de entrenamiento:
   CV MAE: 42558.49 (+/- 368.58)
   CV RMSE: 91127.08 (+/- 1945.63)
   CV R²: 0.8157 (+/- 0.0032)


/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)



3. Métricas en conjunto de TEST:
   MAE: 42380.16
   RMSE: 89909.99
   R²: 0.8206


In [27]:
bestPipeline

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('preprocess',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       Pipeline(steps=[('imputer',
                                                                                                        SimpleImputer(strategy='median')),
                                                                                                       ('scaler',
                                                                                                        StandardScaler())]),
                                                                                       ['banos',
                                                                                        'ap_ppiso',
                                                                                        'dormitorios',
                                                                                        'expensas',
                                                                                        'sup_constru',
                                                                                        'antiguedad',
                                                                                        'ambientes',
                                                                                        'ascensores',
                                                                                        'sup_tot...
                                                                                        'calefacc',
                                                                                        'jardin',
                                                                                        'kitchenette',
                                                                                        'losa_rad',
                                                                                        'parrillero',
                                                                                        'piscina',
                                                                                        'sala_reuniones',
                                                                                        'seguridad',
                                                                                        'amoblado',
                                                                                        'estado',
                                                                                        'orientacion',
                                                                                        'garage',
                                                                                        'comedor'])])),
                                                     ('model',
                                                      HistGradientBoostingRegressor(early_stopping=True,
                                                                                    l2_regularization=5.0,
                                                                                    learning_rate=0.03,
                                                                                    loss='absolute_error',
                                                                                    max_depth=8,
                                                                                    max_iter=400,
                                                                                    random_state=42,
                                                                                    scoring='neg_mean_absolute_error'))]))

In [28]:
bestScore

-42786.05371206288

In [29]:
import joblib
from google.colab import files

expectedColumns = list(X_train.columns)

bundle = {
    "model": bestPipeline,
    "expectedColumns": expectedColumns
}

mae_str = f"{mae_test:.0f}"
rmse_str = f"{rmse_test:.0f}"
filename = f"modelo_inmobiliario_MAE{mae_str}_RMSE{rmse_str}.pkl"

joblib.dump(bundle, filename, protocol=4)
print(f"Modelo guardado como: {filename}")

files.download(filename)


Modelo guardado como: modelo_inmobiliario_MAE42380_RMSE89910.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>